In [180]:
# Import our libraries 

# Pandas and numpy for data wrangling
import pandas as pd 
import numpy as np
# Seaborn / matplotlib for visualization 
import seaborn as sns
import matplotlib as mlt 
# Import the trees from sklearn
from sklearn import tree

# Helper function to split our data
from sklearn.model_selection import train_test_split

# Helper fuctions to evaluate our model.
from sklearn.metrics import  precision_score, accuracy_score, recall_score, confusion_matrix, classification_report, roc_auc_score, f1_score

# Helper function for hyper-parameter turning.


# Import our Decision Tree
from sklearn.tree import DecisionTreeClassifier 

# Import our Random Forest 
from sklearn.ensemble import RandomForestClassifier
# Use inline so our visualizations display in notebook
%matplotlib inline

## Main Steps when building a Machine Learning Model. 
1. Inspect and explore data.
2. Select and engineer features.
3. Build and train model.
4. Evaluate model.

# #1 Inspect and explore data.
* Load titanic data
* Visualize all the data using sns.pairplot
* Check for null values

In [181]:
# Load in the titanic data set.
df_titanic = pd.read_csv('data/titanic.csv', sep = ',')
df_titanic.head()

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [182]:
# Visualize all the data using sns.pairplot
sns.pairplot(df_titanic, hue = 'survived')

In [183]:
# Check for null values
df_titanic.isnull().sum()

passengerid      0
survived         0
pclass           0
name             0
sex              0
age            177
sibsp            0
parch            0
ticket           0
fare             0
cabin          687
embarked         2
dtype: int64

# #2 Select and engineer features.
1. Fill age null values with -999
1. Convert to numerical values if need be by using `pd.get_dummies()`
1. Create a list of the features you are going to use.  In this case use as many or as little as you would like.
1. Define our `X` and `y`
1. Split our data into trainig and testing sets.

In [184]:
# Fill age null values with -999
df_titanic['age'] = df_titanic.age.fillna(-999)

In [185]:
# 1. Convert to numerical values if need be by using `pd.get_dummies()`
dummies = ['embarked','sex','pclass']
df_titanic['pclass'] = df_titanic['pclass'].astype('category')
df_dummies = pd.get_dummies(df_titanic[dummies], dtype = int , )
print(df_dummies.head())
df_titanic[df_dummies.columns] = df_dummies
df_titanic.drop(['cabin','embarked','sex','pclass','passengerid','name','ticket'], axis =1, inplace=True)
df_titanic.head()

   embarked_C  embarked_Q  embarked_S  sex_female  sex_male  pclass_1  \
0           0           0           1           0         1         0   
1           1           0           0           1         0         1   
2           0           0           1           1         0         0   
3           0           0           1           1         0         1   
4           0           0           1           0         1         0   

   pclass_2  pclass_3  
0         0         1  
1         0         0  
2         0         1  
3         0         0  
4         0         1  


,survived,age,sibsp,parch,fare,embarked_C,embarked_Q,embarked_S,sex_female,sex_male,pclass_1,pclass_2,pclass_3
0,0,22.0,1,0,7.2500,0,0,1,0,1,0,0,1
1,1,38.0,1,0,71.2833,1,0,0,1,0,1,0,0
2,1,26.0,0,0,7.9250,0,0,1,1,0,0,0,1
3,1,35.0,1,0,53.1000,0,0,1,1,0,1,0,0
4,0,35.0,0,0,8.0500,0,0,1,0,1,0,0,1


In [186]:
# 2. Create a list of the features we are going to use.
selected_features = df_titanic.drop(['survived'],axis=1).columns


In [187]:
# Define our `X` and `y`
X = df_titanic[selected_features]
y = df_titanic['survived']

In [188]:
# Split our data into trainig and testing sets.
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size= .2, random_state= 42)
print(len(Xtrain), ': ', len(Xtest))


712 :  179


# #3 Build and train model.
1. For our first pass, initialize our model with `max_depth=2`.
2. Fit our model with our training data. 
3. Make predictions of our testing data. 
4. Evaluate and print our model scores using accuracy, precision, recall, f1 scores, and auc scores. 
    * To calculate auc score you have to get the predicted probabilites for the Survived class using `model.predict_proba(X_test)[:,1]`
5. Visualize our Decision Tree using provided code. 


In [189]:
# For our first pass, initialize our model with `max_depth=2`.

model = DecisionTreeClassifier(criterion= 'gini', max_depth=2)

In [190]:
# Fit our model with our training data. 
model.fit(Xtrain,ytrain)


DecisionTreeClassifier(max_depth=2)

In [191]:
# Make predictions of our testing data. 
y_pred = model.predict(Xtest)


In [203]:
# 4. Evaluate and print our model scores using accuracy, precision, recall, f1 scores, and auc scores. 
accuracy = accuracy_score(ytest,y_pred)
print("Accuracy Score: %f" % accuracy)

precision = precision_score(ytest,y_pred)
print("Precision Score: %f" % precision)

recall = recall_score(ytest,y_pred)
print("Recall Score: %f" % recall)

f1 = f1_score(ytest, y_pred)
print('F1 Score: %f' % f1)

# Calculate predicted probabilities
y_pred_proba = model.predict_proba(Xtest)

# Keep only the proba for True
y_pred_proba = y_pred_proba[:,1]

# Compute auc score
auc = roc_auc_score(y_true=ytest, y_score=y_pred_proba)
print('AUC Score: %f' % auc)

feature_imp = pd.DataFrame.from_dict( {'feature_importance': model.feature_importances_,
                                       'feature':selected_features }).sort_values('feature_importance', ascending=False)
feature_imp

Accuracy Score: 0.793296
Precision Score: 0.793651
Recall Score: 0.675676
F1 Score: 0.729927
AUC Score: 0.838481


,feature_importance,feature
7,0.694328,sex_female
11,0.194548,pclass_3
3,0.070196,fare
9,0.040928,pclass_1
0,0.000000,age
1,0.000000,sibsp
2,0.000000,parch
4,0.000000,embarked_C
5,0.000000,embarked_Q
6,0.000000,embarked_S


# Visualize your tree

# Picking the right parameters...

# Parameter tuning of your Decision Tree using GridSearch or RandomizedSearch

### For assistance on this, look at Steves TA Tips code in `TA-Tips/random_forest_tuning.ipynb`


1. Make a dictionary of at least 3 parameters and a list of 3 values for each for your grid search. 
1. Initalize your GridSearchCV with a DecisionTreeClassifier, your param_grid, and what you are optimizing for.  Choose any of the five optimization strategies; accuracy, precision, recall, f1, or roc_auc.
1. Fit your GridSearchCV with your training data. 
1. Print the parameters of your best model. 
1. Evaluate your best model using accuracy, precision, recall, f1 scores, and auc scores. 
1. Visualize your best tree.
1. Which feature was your most important feature?

```python
tree.DecisionTreeClassifier(
    *,
    criterion='gini',
    splitter='best',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    random_state=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    min_impurity_split=None,
    class_weight=None,
    presort='deprecated',
    ccp_alpha=0.0,
)
```


[Tips on how to customize / set the paramters in the decision tree.](https://scikit-learn.org/stable/modules/tree.html#tips-on-practical-use)

In [193]:
# 1. Make a dictionary of at least 3 parameters and a list of 3 values for each for your grid search.from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GridSearchCV
params = { 
   'criterion' : ['gini', 'entropy'],
    'max_features': ['sqrt', 'log2', None], 
    'max_depth': [5, 10], 
    'max_leaf_nodes': [5, 10]}

In [194]:
# 1. Initalize your GridSearchCV with a DecisionTreeClassifier, your param_grid, and what you are optimizing for.  Choose any of the five optimization strategies; accuracy, precision, recall, f1, or roc_auc.
grid_search_cv =  GridSearchCV(model, param_grid=params, cv = 5 )

In [195]:
# 1. Fit your GridSearchCV with your training data. 
grid_search_cv.fit(Xtrain, ytrain)


GridSearchCV(cv=5, estimator=DecisionTreeClassifier(max_depth=2),
             param_grid={'criterion': ['gini', 'entropy'], 'max_depth': [5, 10],
                         'max_features': ['sqrt', 'log2', None],
                         'max_leaf_nodes': [5, 10]})

In [196]:
# 1. Print the parameters of your best model. 
# Print the best parameters it found
print( grid_search_cv.best_estimator_ )

DecisionTreeClassifier(max_depth=5, max_leaf_nodes=5)


In [197]:
# 1. Evaluate your best model using accuracy, precision, recall, f1 scores, and auc scores. 

# This command gives you the best tree
model = grid_search_cv.best_estimator_

# Now lets evaluate our model
y_pred = model.predict(Xtest)

accuracy = accuracy_score(ytest,y_pred)
print("Accuracy Score: %f" % accuracy)

precision = precision_score(ytest,y_pred)
print("Precision Score: %f" % precision)

recall = recall_score(ytest,y_pred)
print("Recall Score: %f" % recall)

f1 = f1_score(ytest, y_pred)
print('F1 Score: %f' % f1)

# Calculate predicted probabilities
y_pred_proba = model.predict_proba(Xtest)[:,1]

# Compute auc score
auc = roc_auc_score(y_true=ytest, y_score=y_pred_proba)
print('AUC Score: %f' % auc)



Accuracy Score: 0.793296
Precision Score: 0.793651
Recall Score: 0.675676
F1 Score: 0.729927
AUC Score: 0.838481


In [198]:
# 1. Which feature was your most important feature?
# Now lets look at our feature importances
feature_imp = pd.DataFrame.from_dict( {'feature_importance': model.feature_importances_,
                                       'feature':selected_features }).sort_values('feature_importance', ascending=False)
feature_imp

,feature_importance,feature
7,0.694328,sex_female
11,0.194548,pclass_3
3,0.070196,fare
9,0.040928,pclass_1
0,0.000000,age
1,0.000000,sibsp
2,0.000000,parch
4,0.000000,embarked_C
5,0.000000,embarked_Q
6,0.000000,embarked_S


# Now onto Random Forests...
Were going to do the same with, but this time with a random forest. Remeber... Repetition is the father of learning.

1. Make a dictionary of at least 3 parameters and a list of 3 values for each for your grid search. 
1. Initalize your GridSearchCV with a RandomForestClassifer, your param_grid, and what you are optimizing for.  Choose any of the five optimization strategies; accuracy, precision, recall, f1, or roc_auc.
1. Fit your GridSearchCV with your training data. 
1. Print the parameters of your best model. 
1. Evaluate your best model using accuracy, precision, recall, f1 scores, and auc scores. 
1. Which feature was your most important feature?


# Parameters of the Random Forest Classifier

```python
RandomForestClassifier(
    n_estimators=100,
    *,
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features='auto',
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    min_impurity_split=None,
    bootstrap=True,
    oob_score=False,
    n_jobs=None,
    random_state=None,
    verbose=0,
    warm_start=False,
    class_weight=None,
    ccp_alpha=0.0,
    max_samples=None,
)
```

In [204]:
# 1. Make a dictionary of at least 3 parameters and a list of 3 values for each for your grid search. 
params = {
    'n_estimators': [50, 100, 200],          
    'max_depth': [None, 10, 20, 30],        
    'min_samples_split': [2, 10, 20],        
    'min_samples_leaf': [1, 5, 10],          #
    'bootstrap': [True, False] 
}

In [205]:
# 1. Initalize your GridSearchCV or RandomizedSearchCV with a RandomForestClassifer, your param_grid, and what you are optimizing for.  Choose any of the five optimization strategies; accuracy, precision, recall, f1, or roc_auc.
grid_search_forest = GridSearchCV(RandomForestClassifier(),param_grid= params,scoring='accuracy')


In [206]:
# 1. Fit your GridSearchCV with your training data. 
grid_search_forest.fit(Xtrain,ytrain)

GridSearchCV(estimator=RandomForestClassifier(),
             param_grid={'bootstrap': [True, False],
                         'max_depth': [None, 10, 20, 30],
                         'min_samples_leaf': [1, 5, 10],
                         'min_samples_split': [2, 10, 20],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy')

In [207]:
# 1. Print the parameters of your best model. 
# Print the best parameters it found
print(grid_search_forest.best_estimator_)




RandomForestClassifier(min_samples_leaf=5, min_samples_split=10,
                       n_estimators=200)


In [209]:
# 1. Evaluate your best model using accuracy, precision, recall, f1 scores, and auc scores. 

# This command gives you tree that has the highest f1-score. 
model = grid_search_cv.best_estimator_


# Now lets evaluate our model
y_pred = model.predict(Xtest)

accuracy = accuracy_score(ytest,y_pred)
print("Accuracy Score: %f" % accuracy)

precision = precision_score(ytest,y_pred)
print("Precision Score: %f" % precision)

recall = recall_score(ytest,y_pred)
print("Recall Score: %f" % recall)

f1 = f1_score(ytest, y_pred)
print('F1 Score: %f' % f1)

# Calculate predicted probabilities
y_pred_proba = model.predict_proba(Xtest)[:,1]

# Compute auc score
auc = roc_auc_score(y_true=ytest, y_score=y_pred_proba)
print('AUC Score: %f' % auc)

Accuracy Score: 0.793296
Precision Score: 0.793651
Recall Score: 0.675676
F1 Score: 0.729927
AUC Score: 0.838481


In [210]:
# 1. Which feature was your most important feature?
# Now lets look at our feature importances
feature_imp = pd.Series(model.feature_importances_,index=selected_features).sort_values(ascending=False)
feature_imp

sex_female    0.694328
pclass_3      0.194548
fare          0.070196
pclass_1      0.040928
age           0.000000
sibsp         0.000000
parch         0.000000
embarked_C    0.000000
embarked_Q    0.000000
embarked_S    0.000000
sex_male      0.000000
pclass_2      0.000000
dtype: float64

# Build a random forest using the ny-vs-sf-housing.csv data. 
* Your target variable, aka the column you are trying to predict, aka your `y` variable is `in_sf`. 
* Can you get an accuracy above %88.8889?
* What was your most important feature?


In [216]:
df = pd.read_csv('data/ny-vs-sf-houses.csv')
df.head()

,in_sf,beds,bath,price,year_built,sqft,price_per_sqft,elevation
0,0,2.0,1.0,999000,1960,1000,999,10
1,0,2.0,2.0,2750000,2006,1418,1939,0
2,0,2.0,2.0,1350000,1900,2150,628,9
3,0,1.0,1.0,629000,1903,500,1258,9
4,0,0.0,1.0,439000,1930,500,878,10


In [219]:
# BUILD, TRAIN, AND EVAULATE A RANDOM FOREST MODEL BELOW. 
X = df.drop(columns=['in_sf'],axis = 1)
y = df['in_sf']
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)
params = {
    'n_estimators': [100, 300,500],          
    'max_depth': [None, 10, 20, 30],        
    'min_samples_split': [2, 10, 20],        
    'min_samples_leaf': [1, 5, 10],          #
    'bootstrap': [True, False] 
}
grid_search_housing = GridSearchCV(RandomForestClassifier(),param_grid=params, scoring='accuracy')
grid_search_housing.fit(X_train,y_train)


GridSearchCV(estimator=RandomForestClassifier(),
             param_grid={'bootstrap': [True, False],
                         'max_depth': [None, 10, 20, 30],
                         'min_samples_leaf': [1, 5, 10],
                         'min_samples_split': [2, 10, 20],
                         'n_estimators': [100, 300, 500]},
             scoring='accuracy')

In [221]:
model = grid_search_housing.best_estimator_
print(model)
y_pred = model.predict(X_test)
print(accuracy_score(y_true=y_test,y_pred=y_pred))


RandomForestClassifier(bootstrap=False, max_depth=10)
0.8943089430894309


# Awesome difficult extra credit below:
Build a classifier using the adult_income.csv data.  
* The target variable is 'class'
* Start with just using these features `selected_features = ['age', 'fnlwgt', 'capital_gain', 'capital_loss', 'hours_per_week']`
* You have to include the pos_label in your precision, recall, and f1 scores. It just tells the classifier which one is the posotive label.  I provided the proper way below.

* See if you can get above 50% f1 score.  
* See some [super tricks and tips here](https://www.kaggle.com/code/jieyima/income-classification-model)

In [ ]:
df = pd.read_csv('data/adult_income.csv')
df.head()